# Stability Criteria: Input Parameters

Analyze slab-side input coverage for Roch and WEAC on ECTP slabs, then export the paper-ready
coverage and attrition figures plus compact tables for the manuscript.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from notebook_utils import create_ectp_slabs, hess_rcparams, load_pits
from paper_figure_utils import (
    build_coverage_comparison_figure,
    build_weac_attrition_figure,
    format_method_path,
    notebook_latex,
    prepare_roch_table,
    prepare_weac_table,
    save_paper_figure,
)
from snowpyt_mechparams.algorithm import find_parameterizations
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig
from snowpyt_mechparams.graph import graph

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(items, **_kwargs):
        return items

hess_rcparams()


## Load Data and Enumerate Paths

Roch coverage is evaluated at the `density` target. WEAC slab-side coverage is evaluated at
`slab_elasticity_parameters` (`E + nu` for every slab layer).


In [2]:
def count_ok(traces, param: str, n_layers: int) -> bool:
    return sum(
        1
        for trace in traces
        if trace.parameter == param
        and trace.layer_index is not None
        and trace.success
        and trace.output is not None
    ) == n_layers


def extract_param_stats(traces, param: str) -> tuple[float, float]:
    successful = [
        trace
        for trace in traces
        if trace.parameter == param
        and trace.layer_index is not None
        and trace.success
        and trace.output is not None
    ]
    if not successful:
        return float('nan'), float('nan')

    nominal_values = []
    relative_uncertainties = []
    for trace in successful:
        output = trace.output
        if hasattr(output, 'nominal_value'):
            nominal = float(output.nominal_value)
            std_dev = float(output.std_dev)
        else:
            nominal = float(output)
            std_dev = 0.0
        nominal_values.append(nominal)
        if nominal != 0:
            relative_uncertainties.append(std_dev / abs(nominal))

    return float(pd.Series(nominal_values).mean()), float(pd.Series(relative_uncertainties).mean())


pits = load_pits()
ectp_slabs = create_ectp_slabs(pits)
total_slabs = len(ectp_slabs)

engine = ExecutionEngine(graph)
roch_paths = find_parameterizations(graph, graph.get_node('density'))
weac_paths = find_parameterizations(graph, graph.get_node('slab_elasticity_parameters'))

print(f'Loaded {len(pits):,} pits and {total_slabs:,} ECTP slabs')
print(f'Roch density pathways: {len(roch_paths)}')
print(f'WEAC slab-elasticity pathways: {len(weac_paths)}')


Loaded 50,278 pits and 14,776 ECTP slabs
Roch density pathways: 4
WEAC slab-elasticity pathways: 32


## Roch Coverage Summary


In [3]:
config = ExecutionConfig(include_method_uncertainty=False)
roch_records = []

for slab in tqdm(ectp_slabs, desc='Roch coverage', unit='slab'):
    results = engine.execute_all(slab, 'density', config=config)
    n_layers = len(slab.layers)
    thickness_ok = all(layer.thickness is not None for layer in slab.layers)

    for pathway in results.pathways.values():
        density_method = pathway.methods_used.get('density', 'data_flow')
        ok_density = count_ok(pathway.computation_trace, 'density', n_layers)
        density_nominal, density_rel_unc = extract_param_stats(pathway.computation_trace, 'density')
        roch_records.append(
            {
                'slab_id': slab.slab_id,
                'density_method': density_method,
                'slab_density_ok': ok_density,
                'thickness_ok': thickness_ok,
                'all_inputs_ok': ok_density and thickness_ok,
                'density_nom': density_nominal,
                'density_rel_unc': density_rel_unc,
            }
        )

roch_df = pd.DataFrame(roch_records)
roch_cov = (
    roch_df.groupby('density_method')
    .agg(
        n_density_ok=('slab_density_ok', 'sum'),
        n_thickness_ok=('thickness_ok', 'sum'),
        n_all_inputs=('all_inputs_ok', 'sum'),
        density_nom=('density_nom', 'mean'),
        density_rel_unc=('density_rel_unc', 'mean'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)
roch_table = prepare_roch_table(roch_cov, total_slabs)
roch_table


Roch coverage: 100%|██████████| 14776/14776 [00:04<00:00, 3641.20slab/s]


,Density method,Successful slabs,Coverage (%),Mean layer density (kg m^-3),Mean relative uncertainty (%)
2,Kim and Jamieson Table 2,"5,951",40.3,177.2,19.1
1,Geldsetzer and Jamieson (2000),"4,539",30.7,162.1,19.4
3,Kim and Jamieson Table 5,"1,145",7.7,159.4,21.4
0,Direct matched density,109,0.7,190.9,10.0


## WEAC Coverage, Attrition, and Figure Export


In [4]:
weac_records = []

for slab in tqdm(ectp_slabs, desc='WEAC slab elasticity', unit='slab'):
    results = engine.execute_all(slab, 'slab_elasticity_parameters', config=config)
    n_layers = len(slab.layers)

    for pathway in results.pathways.values():
        methods = pathway.methods_used
        density_method = methods.get('density', 'data_flow')
        emod_method = methods.get('elastic_modulus', '?')
        nu_method = methods.get('poissons_ratio', '?')

        ok_density = count_ok(pathway.computation_trace, 'density', n_layers)
        ok_emod = count_ok(pathway.computation_trace, 'elastic_modulus', n_layers)
        ok_nu = count_ok(pathway.computation_trace, 'poissons_ratio', n_layers)

        density_nominal, density_rel_unc = extract_param_stats(pathway.computation_trace, 'density')
        emod_nominal, emod_rel_unc = extract_param_stats(pathway.computation_trace, 'elastic_modulus')
        nu_nominal, nu_rel_unc = extract_param_stats(pathway.computation_trace, 'poissons_ratio')

        weac_records.append(
            {
                'slab_id': slab.slab_id,
                'density_method': density_method,
                'emod_method': emod_method,
                'nu_method': nu_method,
                'ok_density': ok_density,
                'ok_emod': ok_emod,
                'ok_nu': ok_nu,
                'all_inputs_ok': ok_density and ok_emod and ok_nu,
                'density_nom': density_nominal,
                'density_rel_unc': density_rel_unc,
                'emod_nom': emod_nominal,
                'emod_rel_unc': emod_rel_unc,
                'nu_nom': nu_nominal,
                'nu_rel_unc': nu_rel_unc,
            }
        )

weac_df = pd.DataFrame(weac_records)
weac_cov = (
    weac_df.groupby(['density_method', 'emod_method', 'nu_method'])
    .agg(
        n_density_ok=('ok_density', 'sum'),
        n_emod_ok=('ok_emod', 'sum'),
        n_nu_ok=('ok_nu', 'sum'),
        n_all_inputs=('all_inputs_ok', 'sum'),
        density_nom=('density_nom', 'mean'),
        density_rel_unc=('density_rel_unc', 'mean'),
        emod_nom=('emod_nom', 'mean'),
        emod_rel_unc=('emod_rel_unc', 'mean'),
        nu_nom=('nu_nom', 'mean'),
        nu_rel_unc=('nu_rel_unc', 'mean'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)

best_density = 'kim_jamieson_table2'
best_emod = 'wautier'
best_nu = 'kochle'
best_subset = weac_df[
    (weac_df['density_method'] == best_density)
    & (weac_df['emod_method'] == best_emod)
    & (weac_df['nu_method'] == best_nu)
].copy()

attrition_steps = [
    ('All ECTP slabs', total_slabs),
    ('Density valid for all slab layers', int(best_subset['ok_density'].sum())),
    ('Density + E valid for all slab layers', int((best_subset['ok_density'] & best_subset['ok_emod']).sum())),
    ('Density + E + nu valid for all slab layers', int(best_subset['all_inputs_ok'].sum())),
]

coverage_paths = save_paper_figure(
    build_coverage_comparison_figure(roch_cov, weac_cov, total_slabs, top_n=12),
    'coverage_comparison',
    close=True,
)
attrition_paths = save_paper_figure(
    build_weac_attrition_figure(
        attrition_steps,
        total_slabs,
        format_method_path(best_density, best_emod, best_nu, short=True),
    ),
    'weac_attrition',
    close=True,
)

weac_table = prepare_weac_table(weac_cov, total_slabs, top_n=8)
print('Coverage figure exports:', coverage_paths)
print('Attrition figure exports:', attrition_paths)
weac_table


WEAC slab elasticity: 100%|██████████| 14776/14776 [01:02<00:00, 235.34slab/s]


Coverage figure exports: {'pdf': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/coverage_comparison.pdf'), 'png': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/coverage_comparison.png')}
Attrition figure exports: {'pdf': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/weac_attrition.pdf'), 'png': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/weac_attrition.png')}


,Density method,E method,nu method,Successful slabs,Coverage (%)
22,Kim and Jamieson Table 2,Wautier et al. (2015),Kochle and Schneebeli (2014),737,5.0
20,Kim and Jamieson Table 2,Schottner et al. (2026),Kochle and Schneebeli (2014),737,5.0
14,Geldsetzer and Jamieson (2000),Wautier et al. (2015),Kochle and Schneebeli (2014),737,5.0
12,Geldsetzer and Jamieson (2000),Schottner et al. (2026),Kochle and Schneebeli (2014),737,5.0
10,Geldsetzer and Jamieson (2000),Kochle and Schneebeli (2014),Kochle and Schneebeli (2014),728,4.9
18,Kim and Jamieson Table 2,Kochle and Schneebeli (2014),Kochle and Schneebeli (2014),627,4.2
16,Kim and Jamieson Table 2,Bergfeld et al. (2023),Kochle and Schneebeli (2014),497,3.4
8,Geldsetzer and Jamieson (2000),Bergfeld et al. (2023),Kochle and Schneebeli (2014),475,3.2


## Copy-Ready LaTeX Tables


In [5]:
print('Roch table:')
print(notebook_latex(roch_table))
print()
print('WEAC table:')
print(notebook_latex(weac_table))


Roch table:
\begin{tabular}{lllll}
\toprule
Density method & Successful slabs & Coverage (%) & Mean layer density (kg m^-3) & Mean relative uncertainty (%) \\
\midrule
Kim and Jamieson Table 2 & 5,951 & 40.3 & 177.2 & 19.1 \\
Geldsetzer and Jamieson (2000) & 4,539 & 30.7 & 162.1 & 19.4 \\
Kim and Jamieson Table 5 & 1,145 & 7.7 & 159.4 & 21.4 \\
Direct matched density & 109 & 0.7 & 190.9 & 10.0 \\
\bottomrule
\end{tabular}


WEAC table:
\begin{tabular}{lllll}
\toprule
Density method & E method & nu method & Successful slabs & Coverage (%) \\
\midrule
Kim and Jamieson Table 2 & Wautier et al. (2015) & Kochle and Schneebeli (2014) & 737 & 5.0 \\
Kim and Jamieson Table 2 & Schottner et al. (2026) & Kochle and Schneebeli (2014) & 737 & 5.0 \\
Geldsetzer and Jamieson (2000) & Wautier et al. (2015) & Kochle and Schneebeli (2014) & 737 & 5.0 \\
Geldsetzer and Jamieson (2000) & Schottner et al. (2026) & Kochle and Schneebeli (2014) & 737 & 5.0 \\
Geldsetzer and Jamieson (2000) & Kochle and Schn